In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import itertools

In [2]:
from src.utils.normalizer import Normalizer

In [3]:
device = "cuda:7"

weight_path = "./models/meta-llama/Llama-2-7b-hf/original_weights/layer_0/self_attn.q_proj.pt"
hessian_diag = weight_path.replace("original_weights", "hessianDiags/seed_0/pajama/128")


weight = torch.load(weight_path, map_location=device)["weight"].to(torch.float32).detach()
hessian_diag = torch.load(hessian_diag, map_location=device)["hessianDiag"].to(torch.float32    )

In [4]:
normalizer, normalized_weight = Normalizer.normalize_init(weight, norm_order=[0,1], zero = [False, False])

N = 2
M = 4

In [5]:
def weighted_l2(weight, original_weight, hessian_diag):
    """
    Computes the regularization loss to penalize the weight to get N:M sparsity
    Args:
        weight (torch.Tensor): The weight tensor to be regularized.
        hessian_diag (torch.Tensor): The diagonal of the Hessian matrix.
        N (int): The numerator of the N:M sparsity ratio.
        M (int): The denominator of the N:M sparsity ratio.
        lambda_ (float): The regularization strength.
    Returns:
        torch.Tensor: The computed regularization loss.
    """
    
    return torch.sum((weight - original_weight) ** 2 * hessian_diag.unsqueeze(0))

In [7]:
importances = torch.abs(normalized_weight * hessian_diag.unsqueeze(0)).view(-1, M)

# get the smallest M-N elements along the last dimension
smallest_idxs = torch.sort(importances, dim=-1).indices[..., :M-N]
largest_idxs = torch.sort(importances, dim=-1).indices[..., -N:]

pruned_weight = torch.zeros_like(normalized_weight).view_as(importances)
retained_values = normalized_weight.view_as(importances)[torch.arange(pruned_weight.size(0)).unsqueeze(1), largest_idxs]
pruned_weight[torch.arange(pruned_weight.size(0)).unsqueeze(1), largest_idxs] = normalized_weight.view_as(importances)[torch.arange(pruned_weight.size(0)).unsqueeze(1), largest_idxs]
pruned_weight = pruned_weight.view_as(normalized_weight)
# mask = torch.ones_like(importances, dtype=torch.bool)
# mask.scatter_(1, smallest_idxs, False)
# pruned_weight = (normalized_weight.view_as(importances) * mask.float()).view_as(normalized_weight)

print(f"original loss: {weighted_l2(pruned_weight, normalized_weight, hessian_diag)}")


original loss: 0.09902780503034592


In [17]:
def permute_pruned_weight(values:torch.Tensor, N:int, M:int):
    """
    Permute the pruned N:M weight tensor to get all the (M choose N) combinations,
    returns a list of len(M choose N) of pruned weights
    """
    idxs_combinations = list(itertools.combinations(range(M), 2))
    
    permuted_weights = []
    for idxs in idxs_combinations:
        permuted_weight = torch.zeros(values.shape[0], M, device=values.device)
        # print(permuted_weight)
        permuted_weight[:, idxs] = values
        permuted_weights.append(permuted_weight)
    return permuted_weights
    
    

In [18]:
permute_pruned_weight(retained_values, N, M)

[tensor([[-0.0305, -0.0094,  0.0000,  0.0000],
         [ 0.0218, -0.0197,  0.0000,  0.0000],
         [ 0.0107,  0.0178,  0.0000,  0.0000],
         ...,
         [ 0.0058, -0.0042,  0.0000,  0.0000],
         [ 0.0111, -0.0234,  0.0000,  0.0000],
         [ 0.0244,  0.0217,  0.0000,  0.0000]], device='cuda:7'),
 tensor([[-0.0305,  0.0000, -0.0094,  0.0000],
         [ 0.0218,  0.0000, -0.0197,  0.0000],
         [ 0.0107,  0.0000,  0.0178,  0.0000],
         ...,
         [ 0.0058,  0.0000, -0.0042,  0.0000],
         [ 0.0111,  0.0000, -0.0234,  0.0000],
         [ 0.0244,  0.0000,  0.0217,  0.0000]], device='cuda:7'),
 tensor([[-0.0305,  0.0000,  0.0000, -0.0094],
         [ 0.0218,  0.0000,  0.0000, -0.0197],
         [ 0.0107,  0.0000,  0.0000,  0.0178],
         ...,
         [ 0.0058,  0.0000,  0.0000, -0.0042],
         [ 0.0111,  0.0000,  0.0000, -0.0234],
         [ 0.0244,  0.0000,  0.0000,  0.0217]], device='cuda:7'),
 tensor([[ 0.0000, -0.0305, -0.0094,  0.0000],
        

In [9]:
import itertools

for comb in itertools.combinations(range(M), 2):
    print(comb)

(0, 1)
(0, 2)
(0, 3)
(1, 2)
(1, 3)
(2, 3)
